In [ ]:
import google.generativeai as genai
import json
import pandas as pd
from time import sleep
from tqdm import tqdm

# Configure Gemini
genai.configure(api_key='')
model = genai.GenerativeModel('gemini-2.0-flash')

def load_real_examples(n=5):
    df = pd.read_json('./Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl', lines=True)
    return df.sample(n=n, random_state=42).to_dict('records')

def create_prompt(real_examples):
    examples_text = ""
    for i, ex in enumerate(real_examples, 1):
        closer = "A" if ex['text_a_is_closer'] else "B"
        examples_text += f"""
Example {i}:
Anchor: {ex['anchor_text']}...
Text A: {ex['text_a']}...
Text B: {ex['text_b']}...
Which is closer: Text {closer}
---
"""
    
    prompt = f"""You are generating training data for a narrative similarity task Sem-Eval 2026 Task.\
      Generate one triple of Wikipedia-style story summaries.\
        We define Narrative similarity by three core similarity components: the abstract theme, the course of action, and the outcomes of a story.

In simple terms, the aspects can be described as follows:

Abstract Theme: The ideas and motives of the story.
Course of Action: The sequence of central events, turning points, etc.
Outcomes: The results of a story.

CRITICAL: Study these REAL examples from the actual task:
{examples_text}

Your generated story MUST:
1. Match Wikipedia plot summary style (factual, encyclopedic tone)
2. Be 150-300 words each
3. Have clear narrative structure (beginning, middle, end)
4. Make similarity judgment based on: theme, plot structure, OR outcome
5. Keep text_a and text_b both somewhat similar to anchor, but one clearly more similar

Generate DIVERSE stories across:
- Different genres
- Different similarity dimensions (sometimes theme matches, sometimes plot, sometimes outcome)
- Different time periods and settings

You don't need to exactly match the examples, but the examples are just to illustrate the format and style and get an understanding of real wikipedia stories.

Output ONLY a valid JSON with one entry in this exact format:
{{
  "triples": [
    {{
      "anchor_text": "full story here",
      "text_a": "full story here",
      "text_b": "full story here", 
      "text_a_is_closer": " true or false based on which text is more similar to the anchor"
    }},  ]
}}

Generate now:"""
    
    return prompt

# Load few-shot examples
real_examples = load_real_examples(n=5)

# Parameters
TOTAL = 1000
BATCH_SIZE = 10
SLEEP_TIME = 5  # seconds between batches

output = []

for batch_start in range(0, TOTAL, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, TOTAL)
    print(f"Generating batch {batch_start}–{batch_end-1}...")
    
    for i in range(batch_start, batch_end):
        try:
            response = model.generate_content(
                create_prompt(real_examples),
                generation_config=genai.types.GenerationConfig(
                    max_output_tokens=4000,
                )
            )

            text = response.text
            # Extract JSON from fenced code block
            if "```json" in text:
                text = text.split("```json")[1].split("```")[0]
            elif "```" in text:
                text = text.split("```")[1].split("```")[0]

            result = json.loads(text.strip())
            output.append(result['triples'][0])

        except Exception as e:
            print(f"⚠️ Error at example {i}: {e}")
            continue
    
    # sleep between batches to avoid hitting rate limits
    sleep(SLEEP_TIME)

print(f"\n✅ Done! Generated {len(output)} examples total.")


Generating batch 0–9...
Generating batch 10–19...
Generating batch 20–29...
Generating batch 30–39...
Generating batch 40–49...
Generating batch 50–59...
Generating batch 60–69...
Generating batch 70–79...
Generating batch 80–89...
Generating batch 90–99...
Generating batch 100–109...
Generating batch 110–119...
Generating batch 120–129...
Generating batch 130–139...
Generating batch 140–149...
⚠️ Error at example 140: Expecting ',' delimiter: line 6 column 575 (char 2046)
Generating batch 150–159...
Generating batch 160–169...
Generating batch 170–179...
Generating batch 180–189...
Generating batch 190–199...
Generating batch 200–209...
Generating batch 210–219...
Generating batch 220–229...
Generating batch 230–239...
Generating batch 240–249...
Generating batch 250–259...
Generating batch 260–269...
Generating batch 270–279...
Generating batch 280–289...
Generating batch 290–299...
Generating batch 300–309...
⚠️ Error at example 307: Expecting ',' delimiter: line 6 column 331 (char 

KeyboardInterrupt: 

In [56]:
output_df = pd.DataFrame(output)
output_path = 'gemini_synthetic_data_p2.jsonl'
output_df.to_json(output_path, orient='records', lines=True)
print(f"\nSaved to: {output_path}")


Saved to: gemini_synthetic_data_p2.jsonl


In [57]:
real_examples

[{'anchor_text': "After a fall from a horse, a wealthy Marquis is believed to be dying. While he lies there, he is comforted by the singing of a beautiful woman. When he unexpectedly recovers, he tries to seek out this young woman. Due to a series of confusions, he believes her to be Empress Eugenie, the wife of Napoleon III of France. In fact, the woman was a Eugenie's hairdresser, a vivacious young woman engaged to be married to an aspiring composer and conductor currently working for the celebrated Jacques Offenbach.",
  'text_a': 'Young David Carroll takes over the publication of a local newspaper in Vermont. Although he is attracted to Dot, "the most sophisticated girl in town," he marries Allie Parker, daughter of the couple who run the boardinghouse where he lives. Allie remains at home when David goes to New York City to sell a musical he has written. There, Dot, now a successful costume designer, uses her influence to get David\'s play produced. David and Dot fall in love, but